# 🚀 Demand Prediction — Pipeline Runner
This notebook is a **thin orchestrator**. All logic lives in the `src/` Python scripts.

| Module | Responsibility |
|---|---|
| `src/config.py` | Paths, hyperparameters, constants |
| `src/data_loader.py` | CSV loading, validation, target split |
| `src/features.py` | Feature engineering & dimensionality reduction |
| `src/model.py` | LightGBM model wrapper |
| `src/pipeline.py` | K-Fold cross-validation loop |
| `src/inference.py` | Test prediction & submission generation |
| `main.py` | Orchestrates all of the above |

---
## Option A — Run the Full Pipeline (One Call)

In [1]:
from main import run_pipeline

results = run_pipeline()

 STEP 1 / 4  —  Loading Data
[DataLoader] Train loaded  — shape (77299, 11)
[DataLoader] Test loaded   — shape (41778, 10)

 STEP 2 / 4  —  Data Summary & Target Split

 Summary: Train
 Shape       : (77299, 11)
 Columns     : ['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']
 Dtypes      :
Index              int64
geohash           object
day                int64
timestamp         object
demand           float64
RoadType          object
NumberofLanes      int64
LargeVehicles     object
Landmarks         object
Temperature      float64
Weather           object
 Null counts :
Index               0
geohash             0
day                 0
timestamp           0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797

[DataLoader] Split target  — X (77299, 10), y (77299,)

 STEP 3 / 4  —  Cross-Validation 

ValueError: Input X contains NaN.
Ridge does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

---
## Option B — Run Step-by-Step
Use this when you want to inspect intermediate outputs between steps.

### Step 1: Load Data

In [1]:
from src.data_loader import load_train, load_test, split_target, summarize

train_df = load_train()
test_df  = load_test()

summarize(train_df, name='Train')
summarize(test_df,  name='Test')

[DataLoader] Train loaded  — shape (77299, 11)
[DataLoader] Test loaded   — shape (41778, 10)

 Summary: Train
 Shape       : (77299, 11)
 Columns     : ['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']
 Dtypes      :
Index              int64
geohash           object
day                int64
timestamp         object
demand           float64
RoadType          object
NumberofLanes      int64
LargeVehicles     object
Landmarks         object
Temperature      float64
Weather           object
 Null counts :
Index               0
geohash             0
day                 0
timestamp           0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797


 Summary: Test
 Shape       : (41778, 10)
 Columns     : ['Index', 'geohash', 'day', 'timestamp', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temp

### Step 2: Split Target

In [2]:
X, y = split_target(train_df)
display(X.head())
display(y.head())

[DataLoader] Split target  — X (77299, 10), y (77299,)


,Index,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,Residential,1,Not Allowed,No,10.803667,Rainy


0    0.048804
1    0.118507
2    0.027132
3    0.003272
4    0.010819
Name: demand, dtype: float64

### Step 3: Cross-Validation

In [3]:
from src.pipeline import ValidationPipeline

pipeline = ValidationPipeline()
ensemble_dict = pipeline.run_cv(X, y)

[Pipeline] Starting 5-Fold Stacking CV...

--- Fold 1/5 ---
Fold 1 - LGB: 0.03899 | XGB: 0.040838 | Ridge: 0.055864

--- Fold 2/5 ---
Fold 2 - LGB: 0.038667 | XGB: 0.040371 | Ridge: 0.057766

--- Fold 3/5 ---
Fold 3 - LGB: 0.036916 | XGB: 0.038742 | Ridge: 0.055827

--- Fold 4/5 ---
Fold 4 - LGB: 0.038124 | XGB: 0.040396 | Ridge: 0.056378

--- Fold 5/5 ---
Fold 5 - LGB: 0.038087 | XGB: 0.039766 | Ridge: 0.057499

[Pipeline] Training Level 2 Meta-Model (XGBoost)...

[Pipeline] Overall Stacking Ensemble OOF Metrics: {'RMSE': np.float64(0.03679), 'MAE': 0.024009}


### Step 4: Test Inference & Save Submission

In [5]:
from src.inference import run_inference

submission = run_inference(ensemble_dict, X, y, test_df)
display(submission.head())

[Inference] Fitting feature engineer on full training data...
[Inference] Transforming test set...
[Inference] Generating Meta Features from Level 1 Models...
[Inference] Predicting final scores using Level 2 Meta-Model...
[Inference] Submission saved to /mnt/e/Flipkart_Gridlock_2.0/DemandPredictionRepo/submissions/baseline_sub.csv


,Index,demand
0,0,0.053333
1,1,0.043241
2,2,0.035452
3,3,0.049723
4,4,0.067342
